# 01 — TBIO8111 sensitive StarDist prior with hybrid TIFF reading

Run this notebook in the StarDist/TensorFlow environment after the hybrid
notebook 00.

It uses OpenSlide for recognized WSI/generic tiled TIFFs and `tifffile` Zarr
region access for valid TIFFs not recognized by OpenSlide. All ten samples
therefore use the same crop, 0.3-µm resizing, sensitive StarDist, and
`tissue-high` QC-gating logic.

```python
prob_thresh = 0.02
nms_thresh = 0.30
```


In [1]:
# FIRST CELL AFTER KERNEL RESTART — before TensorFlow / StarDist imports
import os
import sys

GPU_ID = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"
os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = str(2**40)
os.environ["OPENCV_IO_MAX_IMAGE_WIDTH"] = str(2**24)
os.environ["OPENCV_IO_MAX_IMAGE_HEIGHT"] = str(2**24)

print("Python:", sys.executable)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])


Python: /home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python
CUDA_VISIBLE_DEVICES: 0


In [2]:
# ---------------------------------------------------------------------
# Imports, configuration, and paths
# ---------------------------------------------------------------------
from __future__ import annotations

import sys

import gc
import json
import math
import shutil
import time
import traceback
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import tifffile
import zarr
from csbdeep.utils import normalize
from skimage.segmentation import find_boundaries
from stardist.models import StarDist2D

try:
    import openslide
except Exception as exc:
    raise ImportError(
        "Notebook 01 requires openslide-python and the OpenSlide native library."
    ) from exc

CONFIG_PATH = Path(
    os.environ.get(
        "VISIUMHD_PIPELINE_CONFIG",
        "/stash/data/nonclin/TBIO-8111_VisiumHD-HELIOS-CA1201001-NSCLC/"
        "derived_files/tbio8110_stardist_proseg_resolvi_v1/"
        "00_config/pipeline_config.json",
    )
)
if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Run notebook 00 first or update CONFIG_PATH: {CONFIG_PATH}"
    )

CONFIG = json.loads(CONFIG_PATH.read_text())
CUSTOM_FUNCTION_DIR = Path(CONFIG["paths"]["functiondirs"])
if CUSTOM_FUNCTION_DIR.exists():
    if str(CUSTOM_FUNCTION_DIR) not in sys.path:
        sys.path.insert(0, str(CUSTOM_FUNCTION_DIR))
    print("Custom function directory enabled:", CUSTOM_FUNCTION_DIR)
else:
    warnings.warn(
        f"Configured functiondirs path does not exist: {CUSTOM_FUNCTION_DIR}. "
        "The built-in notebook helpers will be used."
    )
DERIVED_ROOT = Path(CONFIG["paths"]["derived_root"])
TEMP_ROOT = Path(CONFIG["paths"]["temp_root"])
CONFIG_ROOT = Path(CONFIG["paths"]["config_root"])
MANIFEST_PATH = CONFIG_ROOT / "sample_manifest.csv"
manifest = pd.read_csv(MANIFEST_PATH)

PRIOR_TEMP_ROOT = TEMP_ROOT / "01_stardist_qcprior"
PRIOR_DERIVED_ROOT = DERIVED_ROOT / "01_stardist_qcprior"
PRIOR_TEMP_ROOT.mkdir(parents=True, exist_ok=True)
PRIOR_DERIVED_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_MPP = 0.3
CROP_BUFFER_UM = 50.0
STARDIST_MODEL = "2D_versatile_he"
PROB_THRESH = 0.02
NMS_THRESH = 0.30
MIN_OVERLAP = 256
CONTEXT = 128
MIN_QC_KEEP_BINS_PER_NUCLEUS = 1

STARDIST_INFERENCE_ATTEMPTS = [
    {
        "name": "block4096_tiles2x2",
        "block_size": 4096,
        "n_tiles": (2, 2, 1),
    },
    {
        "name": "block2048_tiles2x2",
        "block_size": 2048,
        "n_tiles": (2, 2, 1),
    },
    {
        "name": "block2048_tiles4x4",
        "block_size": 2048,
        "n_tiles": (4, 4, 1),
    },
]

MASK_STATS_CHUNK_ROWS = 256
USE_EXISTING_COMPLETE = True
OVERWRITE = False
CONTINUE_ON_ERROR = True
PREVIEW_MAX_SIDE = 3500
RANDOM_SEED = int(CONFIG["safeguards"]["random_seed"])

PIPELINE_VERSION = "reusable-hybrid-tiff-stardist-qcprior-v2"
PARAMETER_PROFILE = "2D_versatile_he_prob0p02_nms0p30_qc_gated"

print("TensorFlow:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError("TensorFlow does not see a GPU.")
print("Samples:", manifest["sample"].astype(str).tolist())


Custom function directory enabled: /host_root/nethome/reny28/Projects/Custom_functions/python_functions
TensorFlow: 2.21.0
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Samples: ['C2D15_14_60', 'C2D15_18_68', 'C2D15_22_24', 'C2D15_30_81', 'C2D15_7_93', 'Screen_14_60', 'Screen_18_68', 'Screen_22_24', 'Screen_30_81', 'Screen_7_93']


In [3]:
# ---------------------------------------------------------------------
# Path and image helpers
# ---------------------------------------------------------------------
def paths_for_sample(row: pd.Series) -> dict[str, Path]:
    sample = str(row["sample"])
    temp = PRIOR_TEMP_ROOT / sample
    durable = PRIOR_DERIVED_ROOT / sample
    temp.mkdir(parents=True, exist_ok=True)
    durable.mkdir(parents=True, exist_ok=True)

    return {
        "sample": sample,
        "image": Path(row["image_path"]),
        "aligned_qc": Path(row["aligned_qc_parquet"]),
        "crop_tiff": temp / f"{sample}_he_crop_0p3mpp.tiff",
        "raw_mask": temp / f"{sample}_stardist_raw_uint32.npy",
        "clean_mask": temp / f"{sample}_stardist_qcfiltered_uint32.npy",
        "audit": durable / f"{sample}_stardist_qc_label_audit.csv.gz",
        "metadata": durable / f"{sample}_stardist_qc_metadata.json",
        "overview": durable / f"{sample}_stardist_qc_overview.png",
        "center_preview": durable / f"{sample}_stardist_qc_center.png",
        "success": durable / f"{sample}_stardist_qc_SUCCESS.json",
        "failure": durable / f"{sample}_stardist_qc_FAILURE.json",
    }


def atomic_json(payload, path: Path):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, default=str))
    temporary.replace(path)


def _select_largest_tiff_series(tif):
    candidates = []
    for index, series in enumerate(tif.series):
        axes = str(series.axes).upper()
        if "Y" not in axes or "X" not in axes:
            continue
        shape = tuple(int(v) for v in series.shape)
        height = shape[axes.index("Y")]
        width = shape[axes.index("X")]
        candidates.append((height * width, index, series))

    if not candidates:
        raise ValueError("No TIFF series with X/Y axes was found.")

    _, index, series = max(candidates, key=lambda item: item[0])
    return int(index), series


def _array_to_rgb_uint8(array, axes):
    axes = str(axes).upper()
    data = np.asarray(array)

    if "Y" not in axes or "X" not in axes:
        raise ValueError(
            f"Cannot interpret TIFF axes={axes!r}, shape={data.shape}."
        )

    y_index = axes.index("Y")
    x_index = axes.index("X")
    channel_axis = next(
        (
            axes.index(axis)
            for axis in ("S", "C")
            if axis in axes
        ),
        None,
    )

    if channel_axis is None:
        data = np.moveaxis(data, (y_index, x_index), (0, 1))
        data = np.squeeze(data)
        data = np.repeat(data[..., None], 3, axis=-1)
    else:
        data = np.moveaxis(
            data,
            (y_index, x_index, channel_axis),
            (0, 1, 2),
        )
        data = np.squeeze(data)
        if data.ndim == 2:
            data = np.repeat(data[..., None], 3, axis=-1)
        elif data.shape[-1] == 1:
            data = np.repeat(data, 3, axis=-1)
        else:
            data = data[..., :3]

    if data.dtype == np.uint8:
        return np.ascontiguousarray(data)

    data = data.astype(np.float32, copy=False)
    output = np.zeros(data.shape, dtype=np.uint8)

    for channel in range(data.shape[-1]):
        values = data[..., channel]
        finite = np.isfinite(values)
        if not finite.any():
            continue
        low, high = np.percentile(values[finite], [0.5, 99.5])
        if high <= low:
            high = low + 1.0
        output[..., channel] = np.clip(
            (values - low) / (high - low) * 255.0,
            0,
            255,
        ).astype(np.uint8)

    return output


def read_tifffile_region_rgb(
    image_path: Path,
    left: int,
    top: int,
    width: int,
    height: int,
) -> np.ndarray:
    """
    Read a level-0 TIFF rectangle through tifffile's Zarr interface.

    Only the requested region is materialized as a NumPy array.
    """
    right = int(left + width)
    bottom = int(top + height)

    with tifffile.TiffFile(image_path) as tif:
        _, series = _select_largest_tiff_series(tif)
        level = list(getattr(series, "levels", [series]))[0]
        axes = str(level.axes).upper()

        store = level.aszarr()
        try:
            array = zarr.open(store, mode="r")
            selection = []
            remaining_axes = []

            for axis in axes:
                if axis == "Y":
                    selection.append(slice(int(top), bottom))
                    remaining_axes.append(axis)
                elif axis == "X":
                    selection.append(slice(int(left), right))
                    remaining_axes.append(axis)
                elif axis in {"S", "C"}:
                    selection.append(slice(None))
                    remaining_axes.append(axis)
                else:
                    selection.append(0)

            crop = np.asarray(array[tuple(selection)])
        finally:
            close = getattr(store, "close", None)
            if callable(close):
                close()

    return _array_to_rgb_uint8(
        crop,
        "".join(remaining_axes),
    )


def read_image_region_rgb(
    image_path: Path,
    left: int,
    top: int,
    width: int,
    height: int,
    *,
    reader_backend: str | None,
) -> np.ndarray:
    backend = (
        str(reader_backend)
        if reader_backend is not None
        else ""
    ).lower()

    if backend == "openslide":
        with openslide.OpenSlide(str(image_path)) as slide:
            rgba = slide.read_region(
                (int(left), int(top)),
                0,
                (int(width), int(height)),
            ).convert("RGBA")

        from PIL import Image

        background = Image.new(
            "RGBA",
            rgba.size,
            (255, 255, 255, 255),
        )
        background.alpha_composite(rgba)
        return np.asarray(
            background.convert("RGB"),
            dtype=np.uint8,
        )

    if backend == "tifffile":
        return read_tifffile_region_rgb(
            image_path,
            left,
            top,
            width,
            height,
        )

    # Backward-compatible auto-detection for older manifests.
    detected = openslide.OpenSlide.detect_format(str(image_path))
    if detected is not None:
        return read_image_region_rgb(
            image_path,
            left,
            top,
            width,
            height,
            reader_backend="openslide",
        )

    return read_tifffile_region_rgb(
        image_path,
        left,
        top,
        width,
        height,
    )


def calculate_crop(aligned: pd.DataFrame, row: pd.Series):
    retained = aligned[
        aligned["qc_keep"].astype(bool)
        & aligned["image_in_bounds"].astype(bool)
    ]
    if retained.empty:
        raise RuntimeError("No retained QC bins are inside the image.")

    mpp_x = float(row["image_mpp_x"])
    mpp_y = float(row["image_mpp_y"])
    buffer_x = int(math.ceil(CROP_BUFFER_UM / mpp_x))
    buffer_y = int(math.ceil(CROP_BUFFER_UM / mpp_y))

    left = max(0, int(math.floor(retained["image_x_px"].min())) - buffer_x)
    right = min(
        int(row["image_width"]),
        int(math.ceil(retained["image_x_px"].max())) + buffer_x + 1,
    )
    top = max(0, int(math.floor(retained["image_y_px"].min())) - buffer_y)
    bottom = min(
        int(row["image_height"]),
        int(math.ceil(retained["image_y_px"].max())) + buffer_y + 1,
    )
    if right <= left or bottom <= top:
        raise ValueError((left, top, right, bottom))
    return left, top, right, bottom


def resize_crop_to_target_mpp(
    crop: np.ndarray,
    source_mpp_x: float,
    source_mpp_y: float,
):
    output_width = max(
        1,
        int(round(crop.shape[1] * source_mpp_x / TARGET_MPP)),
    )
    output_height = max(
        1,
        int(round(crop.shape[0] * source_mpp_y / TARGET_MPP)),
    )
    resized = cv2.resize(
        crop,
        (output_width, output_height),
        interpolation=cv2.INTER_AREA
        if output_width <= crop.shape[1] and output_height <= crop.shape[0]
        else cv2.INTER_CUBIC,
    )
    return np.asarray(resized, dtype=np.uint8)


def image_to_mask_coordinates(
    aligned: pd.DataFrame,
    *,
    crop_left: int,
    crop_top: int,
    image_mpp_x: float,
    image_mpp_y: float,
):
    mask_col = (
        (aligned["image_x_px"].to_numpy(dtype=np.float64) - crop_left)
        * image_mpp_x
        / TARGET_MPP
    )
    mask_row = (
        (aligned["image_y_px"].to_numpy(dtype=np.float64) - crop_top)
        * image_mpp_y
        / TARGET_MPP
    )
    return np.column_stack([mask_row, mask_col])


In [4]:
# ---------------------------------------------------------------------
# StarDist GPU-inference helpers
# ---------------------------------------------------------------------
def is_gpu_oom(exc: BaseException) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return (
        isinstance(exc, tf.errors.ResourceExhaustedError)
        or "out of memory" in text
        or "resourceexhausted" in text
        or "oom when allocating" in text
    )


def run_stardist_big(
    model,
    normalized_image: np.ndarray,
    *,
    block_size: int,
    n_tiles: tuple[int, int, int],
):
    kwargs = dict(
        axes="YXC",
        block_size=int(block_size),
        min_overlap=int(MIN_OVERLAP),
        context=int(CONTEXT),
        prob_thresh=float(PROB_THRESH),
        nms_thresh=float(NMS_THRESH),
        n_tiles=tuple(int(v) for v in n_tiles),
    )
    print("StarDist inference settings:", kwargs)
    try:
        return model.predict_instances_big(
            normalized_image,
            show_progress=True,
            show_tile_progress=False,
            **kwargs,
        )
    except TypeError as exc:
        if "unexpected keyword" not in str(exc).lower():
            raise
        return model.predict_instances_big(normalized_image, **kwargs)


def run_stardist_with_fallback(normalized_image: np.ndarray):
    failures = []
    for attempt in STARDIST_INFERENCE_ATTEMPTS:
        model = None
        try:
            print("\nAttempt:", attempt["name"])
            model = StarDist2D.from_pretrained(STARDIST_MODEL)
            labels, details = run_stardist_big(
                model,
                normalized_image,
                block_size=attempt["block_size"],
                n_tiles=attempt["n_tiles"],
            )
            used = {
                "name": attempt["name"],
                "block_size": int(attempt["block_size"]),
                "n_tiles": list(map(int, attempt["n_tiles"])),
            }
            return labels, details, used, failures
        except Exception as exc:
            record = {
                "name": attempt["name"],
                "error": f"{type(exc).__name__}: {exc}",
                "is_gpu_oom": bool(is_gpu_oom(exc)),
            }
            failures.append(record)
            if not is_gpu_oom(exc):
                raise
            print("GPU OOM; retrying with a safer configuration.")
            print(json.dumps(record, indent=2))
        finally:
            if model is not None:
                del model
            tf.keras.backend.clear_session()
            gc.collect()

    raise RuntimeError(
        "All StarDist inference attempts failed:\n"
        + json.dumps(failures, indent=2)
    )


In [5]:
# ---------------------------------------------------------------------
# Label statistics, QC cleanup, affine fitting, and previews
# ---------------------------------------------------------------------
def dense_label_statistics(mask: np.ndarray, max_label: int):
    pixel_count = np.zeros(max_label + 1, dtype=np.int64)
    row_sum = np.zeros(max_label + 1, dtype=np.float64)
    col_sum = np.zeros(max_label + 1, dtype=np.float64)

    for start in range(0, mask.shape[0], MASK_STATS_CHUNK_ROWS):
        stop = min(start + MASK_STATS_CHUNK_ROWS, mask.shape[0])
        chunk = np.asarray(mask[start:stop])
        rr, cc = np.nonzero(chunk)
        if len(rr):
            labels = chunk[rr, cc].astype(np.int64, copy=False)
            pixel_count += np.bincount(labels, minlength=max_label + 1)
            row_sum += np.bincount(
                labels,
                weights=(rr + start).astype(np.float64),
                minlength=max_label + 1,
            )
            col_sum += np.bincount(
                labels,
                weights=cc.astype(np.float64),
                minlength=max_label + 1,
            )

    centroid_row = np.full(max_label + 1, np.nan)
    centroid_col = np.full(max_label + 1, np.nan)
    positive = pixel_count > 0
    centroid_row[positive] = row_sum[positive] / pixel_count[positive]
    centroid_col[positive] = col_sum[positive] / pixel_count[positive]
    return pixel_count, centroid_row, centroid_col


def sample_mask(mask: np.ndarray, coords_rc: np.ndarray):
    rows = np.rint(coords_rc[:, 0]).astype(np.int64)
    cols = np.rint(coords_rc[:, 1]).astype(np.int64)
    in_bounds = (
        (rows >= 0)
        & (rows < mask.shape[0])
        & (cols >= 0)
        & (cols < mask.shape[1])
    )
    sampled = np.zeros(len(rows), dtype=np.uint32)
    sampled[in_bounds] = mask[rows[in_bounds], cols[in_bounds]]
    return sampled, in_bounds


def fit_mask_to_proseg_affine(
    coords_rc: np.ndarray,
    aligned: pd.DataFrame,
    in_bounds: np.ndarray,
):
    valid = (
        in_bounds
        & np.isfinite(aligned["proseg_x_um"].to_numpy())
        & np.isfinite(aligned["proseg_y_um"].to_numpy())
    )
    indices = np.flatnonzero(valid)
    if len(indices) < 100:
        raise RuntimeError("Too few valid bins for affine fitting.")

    rng = np.random.default_rng(RANDOM_SEED)
    fit_indices = (
        rng.choice(indices, 250_000, replace=False)
        if len(indices) > 250_000
        else indices
    )
    design = np.column_stack(
        [
            coords_rc[fit_indices, 1],
            coords_rc[fit_indices, 0],
            np.ones(len(fit_indices)),
        ]
    )
    target_x = aligned["proseg_x_um"].to_numpy(dtype=np.float64)
    target_y = aligned["proseg_y_um"].to_numpy(dtype=np.float64)
    x_coef, *_ = np.linalg.lstsq(
        design, target_x[fit_indices], rcond=None
    )
    y_coef, *_ = np.linalg.lstsq(
        design, target_y[fit_indices], rcond=None
    )

    validation_indices = (
        rng.choice(indices, min(250_000, len(indices)), replace=False)
        if len(indices) > 250_000
        else indices
    )
    validation_design = np.column_stack(
        [
            coords_rc[validation_indices, 1],
            coords_rc[validation_indices, 0],
            np.ones(len(validation_indices)),
        ]
    )
    predicted = np.column_stack(
        [validation_design @ x_coef, validation_design @ y_coef]
    )
    observed = np.column_stack(
        [target_x[validation_indices], target_y[validation_indices]]
    )
    errors = np.linalg.norm(predicted - observed, axis=1)

    report = {
        "x_transform": x_coef.tolist(),
        "y_transform": y_coef.tolist(),
        "validation_rmse_um": float(np.sqrt(np.mean(errors**2))),
        "validation_median_error_um": float(np.median(errors)),
        "validation_p99_error_um": float(np.quantile(errors, 0.99)),
        "validation_max_error_um": float(np.max(errors)),
        "n_fit_points": int(len(fit_indices)),
        "n_validation_points": int(len(validation_indices)),
    }
    if report["validation_p99_error_um"] > 0.75:
        raise RuntimeError(
            "Mask-to-Proseg affine residual is unexpectedly large: "
            + json.dumps(report, indent=2)
        )
    return report


def save_overlay(crop, raw_mask, clean_mask, output_path, title):
    stride = max(1, int(np.ceil(max(crop.shape[:2]) / PREVIEW_MAX_SIDE)))
    image_small = np.asarray(crop[::stride, ::stride])
    raw_small = raw_mask[::stride, ::stride]
    clean_small = clean_mask[::stride, ::stride]
    raw_boundary = find_boundaries(raw_small, mode="outer")
    clean_boundary = find_boundaries(clean_small, mode="outer")

    fig, ax = plt.subplots(figsize=(12, 12))
    ax.imshow(image_small)
    ax.contour(
        raw_boundary.astype(float),
        levels=[0.5],
        colors=["#FF3366"],
        linewidths=0.18,
        alpha=0.45,
    )
    ax.contour(
        clean_boundary.astype(float),
        levels=[0.5],
        colors=["#00E5FF"],
        linewidths=0.28,
        alpha=0.85,
    )
    ax.set_title(title + "\nred=all sensitive detections; cyan=QC retained")
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


In [6]:
# ---------------------------------------------------------------------
# Process one sample
# ---------------------------------------------------------------------
def process_sample(row: pd.Series) -> dict:
    sample = str(row["sample"])
    p = paths_for_sample(row)

    expected_profile = {
        "pipeline_version": PIPELINE_VERSION,
        "parameter_profile": PARAMETER_PROFILE,
        "prob_thresh": PROB_THRESH,
        "nms_thresh": NMS_THRESH,
        "target_mpp": TARGET_MPP,
        "minimum_qc_keep_bins_per_nucleus": (
            MIN_QC_KEEP_BINS_PER_NUCLEUS
        ),
    }

    if (
        USE_EXISTING_COMPLETE
        and p["success"].exists()
        and p["clean_mask"].exists()
        and p["metadata"].exists()
        and not OVERWRITE
    ):
        existing = json.loads(p["success"].read_text())
        if all(existing.get(k) == v for k, v in expected_profile.items()):
            print("Reusing parameter-matched prior:", sample)
            return existing
        print("Existing prior does not match requested profile; rerunning.")

    p["success"].unlink(missing_ok=True)

    aligned = pd.read_parquet(p["aligned_qc"])
    left, top, right, bottom = calculate_crop(aligned, row)
    reader_backend = (
        row.get("image_reader_backend")
        if "image_reader_backend" in row.index
        else None
    )
    print("Image reader backend:", reader_backend)

    source_crop = read_image_region_rgb(
        p["image"],
        left,
        top,
        right - left,
        bottom - top,
        reader_backend=reader_backend,
    )
    crop = resize_crop_to_target_mpp(
        source_crop,
        float(row["image_mpp_x"]),
        float(row["image_mpp_y"]),
    )
    del source_crop
    gc.collect()

    print(
        sample,
        "scaled crop:",
        crop.shape,
        "source mpp:",
        (float(row["image_mpp_x"]), float(row["image_mpp_y"])),
    )
    tifffile.imwrite(
        p["crop_tiff"],
        crop,
        photometric="rgb",
        bigtiff=True,
        compression="zlib",
    )

    normalized = normalize(crop, 1, 99.8, axis=(0, 1))
    started = time.time()
    raw_labels, details, used_attempt, failed_attempts = (
        run_stardist_with_fallback(normalized)
    )
    elapsed = time.time() - started
    raw_labels = np.asarray(raw_labels, dtype=np.uint32)
    np.save(p["raw_mask"], raw_labels)

    coords_rc = image_to_mask_coordinates(
        aligned,
        crop_left=left,
        crop_top=top,
        image_mpp_x=float(row["image_mpp_x"]),
        image_mpp_y=float(row["image_mpp_y"]),
    )
    sampled, in_bounds = sample_mask(raw_labels, coords_rc)
    keep_bins = aligned["qc_keep"].to_numpy(dtype=bool)

    keep_in_bounds = float(in_bounds[keep_bins].mean())
    if keep_in_bounds < 0.95:
        raise RuntimeError(
            f"Only {keep_in_bounds:.3%} of retained bins map inside the crop."
        )

    max_label = int(raw_labels.max(initial=0))
    qc_keep_counts = np.bincount(
        sampled[in_bounds & keep_bins].astype(np.int64),
        minlength=max_label + 1,
    )
    qc_drop_counts = np.bincount(
        sampled[in_bounds & ~keep_bins].astype(np.int64),
        minlength=max_label + 1,
    )

    pixel_count, centroid_row, centroid_col = dense_label_statistics(
        raw_labels,
        max_label,
    )
    raw_ids = np.flatnonzero(pixel_count > 0)
    raw_ids = raw_ids[raw_ids != 0]
    retain = (
        qc_keep_counts[raw_ids]
        >= MIN_QC_KEEP_BINS_PER_NUCLEUS
    )

    retained_raw_ids = raw_ids[retain]
    relabel_lut = np.zeros(max_label + 1, dtype=np.uint32)
    relabel_lut[retained_raw_ids] = np.arange(
        1,
        len(retained_raw_ids) + 1,
        dtype=np.uint32,
    )
    clean_labels = relabel_lut[raw_labels]
    np.save(p["clean_mask"], clean_labels)

    audit = pd.DataFrame(
        {
            "raw_label_id": raw_ids.astype(np.int64),
            "clean_label_id": relabel_lut[raw_ids].astype(np.int64),
            "n_mask_pixels": pixel_count[raw_ids],
            "centroid_row_px": centroid_row[raw_ids],
            "centroid_col_px": centroid_col[raw_ids],
            "n_qc_keep_bins": qc_keep_counts[raw_ids],
            "n_qc_drop_bins": qc_drop_counts[raw_ids],
            "n_qc_sampled_bins": (
                qc_keep_counts[raw_ids] + qc_drop_counts[raw_ids]
            ),
            "retain_for_proseg": retain,
        }
    )
    audit.to_csv(p["audit"], index=False, compression="gzip")

    affine_report = fit_mask_to_proseg_affine(
        coords_rc,
        aligned,
        in_bounds,
    )

    metadata = {
        "sample": sample,
        "source_image": str(p["image"]),
        "image_reader_backend": str(reader_backend),
        "image_crop": str(p["crop_tiff"]),
        "raw_mask": str(p["raw_mask"]),
        "clean_mask": str(p["clean_mask"]),
        "audit": str(p["audit"]),
        "source_image_mpp_x": float(row["image_mpp_x"]),
        "source_image_mpp_y": float(row["image_mpp_y"]),
        "target_mpp": TARGET_MPP,
        "crop_level0_bounds": [left, top, right, bottom],
        "crop_shape_target_mpp": list(map(int, crop.shape)),
        "stardist_model": STARDIST_MODEL,
        "prob_thresh": PROB_THRESH,
        "nms_thresh": NMS_THRESH,
        "minimum_qc_keep_bins_per_nucleus": (
            MIN_QC_KEEP_BINS_PER_NUCLEUS
        ),
        "parameter_profile": PARAMETER_PROFILE,
        "inference_attempt_used": used_attempt,
        "failed_inference_attempts": failed_attempts,
        "n_raw_objects": int(len(raw_ids)),
        "n_retained_objects": int(retain.sum()),
        "n_removed_objects": int((~retain).sum()),
        "retained_qc_bin_in_bounds_fraction": keep_in_bounds,
        "elapsed_seconds": elapsed,
        **affine_report,
        "pipeline_version": PIPELINE_VERSION,
    }
    atomic_json(metadata, p["metadata"])

    save_overlay(
        crop,
        raw_labels,
        clean_labels,
        p["overview"],
        f"{sample}: sensitive StarDist plus 2-µm QC gate",
    )

    # Center crop at native target resolution for close inspection.
    preview_size = min(4000, crop.shape[0], crop.shape[1])
    upper = max(0, (crop.shape[0] - preview_size) // 2)
    left_preview = max(0, (crop.shape[1] - preview_size) // 2)
    save_overlay(
        crop[upper : upper + preview_size, left_preview : left_preview + preview_size],
        raw_labels[upper : upper + preview_size, left_preview : left_preview + preview_size],
        clean_labels[upper : upper + preview_size, left_preview : left_preview + preview_size],
        p["center_preview"],
        f"{sample}: center crop",
    )

    success = {
        **expected_profile,
        "sample": sample,
        "completed": True,
        "clean_mask": str(p["clean_mask"]),
        "metadata": str(p["metadata"]),
        "audit": str(p["audit"]),
        "n_raw_objects": int(len(raw_ids)),
        "n_retained_objects": int(retain.sum()),
        "inference_attempt_used": used_attempt,
    }
    atomic_json(success, p["success"])
    p["failure"].unlink(missing_ok=True)

    del crop, normalized, raw_labels, clean_labels, sampled, audit
    gc.collect()
    return success


In [ ]:
# ---------------------------------------------------------------------
# Run selected samples sequentially
# ---------------------------------------------------------------------
results = {}
failures = {}

for _, row in manifest.iterrows():
    sample = str(row["sample"])
    print("\n" + "=" * 90)
    print("StarDist sample:", sample)
    try:
        results[sample] = process_sample(row)
    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"
        failures[sample] = error
        p = paths_for_sample(row)
        atomic_json(
            {"sample": sample, "error": error},
            p["failure"],
        )
        traceback.print_exc(limit=25)
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        tf.keras.backend.clear_session()
        gc.collect()

summary = pd.DataFrame(results.values())
summary.to_csv(
    PRIOR_DERIVED_ROOT / "all_samples_stardist_qcprior_summary.csv",
    index=False,
)
(
    PRIOR_DERIVED_ROOT / "all_samples_stardist_qcprior_failures.json"
).write_text(json.dumps(failures, indent=2))

print("Completed:", sorted(results))
print("Failures:", json.dumps(failures, indent=2))

if failures:
    raise RuntimeError(
        "At least one StarDist prior failed. Successful samples remain reusable."
    )



StarDist sample: C2D15_14_60
Image reader backend: openslide
C2D15_14_60 scaled crop: (22717, 22864, 3) source mpp: (0.220147939415287, 0.2201285550761644)

Attempt: block4096_tiles2x2
Found model '2D_versatile_he' for 'StarDist2D'.


W0000 00:00:1788311425.334290    4080 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.


Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
StarDist inference settings: {'axes': 'YXC', 'block_size': 4096, 'min_overlap': 256, 'context': 128, 'prob_thresh': 0.02, 'nms_thresh': 0.3, 'n_tiles': (2, 2, 1)}
effective: block_size=(4096, 4096, 3), min_overlap=(256, 256, 0), context=(128, 128, 0)
changing 'show_tile_progress' from False to False


  0%|          | 0/49 [00:00<?, ?it/s]E0000 00:00:1788311437.004294    4689 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1788311437.799581    4689 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
W0000 00:00:1788311437.980288    4689 bfc_allocator.cc:383] Garbage collection: deallocate free memory regions (i.e., allocations) so that we can re-allocate a larger region to avoid OOM due to memory fragmentation. If you see this message frequently, you are running near the threshold of the available device memory and re-allocation may incur great performance overhead. You may try smaller batch sizes to observe the performance impact. Set TF_ENABLE_GPU_GARBAGE_COLLECTION=false if you'd like to disable this feature.
E0000 00:00:1788311439.945579    4689 cud


StarDist sample: C2D15_18_68
Image reader backend: openslide
C2D15_18_68 scaled crop: (22781, 22683, 3) source mpp: (0.220147939415287, 0.2201285550761644)

Attempt: block4096_tiles2x2
Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
StarDist inference settings: {'axes': 'YXC', 'block_size': 4096, 'min_overlap': 256, 'context': 128, 'prob_thresh': 0.02, 'nms_thresh': 0.3, 'n_tiles': (2, 2, 1)}
effective: block_size=(4096, 4096, 3), min_overlap=(256, 256, 0), context=(128, 128, 0)
changing 'show_tile_progress' from False to False


...

100%|██████████| 49/49 [01:43<00:00,  2.11s/it]



StarDist sample: Screen_30_81
Image reader backend: openslide
Screen_30_81 scaled crop: (22447, 22059, 3) source mpp: (0.220147939415287, 0.2201285550761644)

Attempt: block4096_tiles2x2
Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
StarDist inference settings: {'axes': 'YXC', 'block_size': 4096, 'min_overlap': 256, 'context': 128, 'prob_thresh': 0.02, 'nms_thresh': 0.3, 'n_tiles': (2, 2, 1)}
effective: block_size=(4096, 4096, 3), min_overlap=(256, 256, 0), context=(128, 128, 0)
changing 'show_tile_progress' from False to False


100%|██████████| 49/49 [01:46<00:00,  2.17s/it]



StarDist sample: Screen_7_93
Image reader backend: openslide
Screen_7_93 scaled crop: (21062, 22044, 3) source mpp: (0.220147939415287, 0.2201285550761644)

Attempt: block4096_tiles2x2
Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
StarDist inference settings: {'axes': 'YXC', 'block_size': 4096, 'min_overlap': 256, 'context': 128, 'prob_thresh': 0.02, 'nms_thresh': 0.3, 'n_tiles': (2, 2, 1)}
effective: block_size=(4096, 4096, 3), min_overlap=(256, 256, 0), context=(128, 128, 0)
changing 'show_tile_progress' from False to False


100%|██████████| 42/42 [01:22<00:00,  1.97s/it]


## Required visual review

For every sample, inspect:

```text
01_stardist_qcprior/<sample>/<sample>_stardist_qc_overview.png
01_stardist_qcprior/<sample>/<sample>_stardist_qc_center.png
```

Sensitive detections are red and retained objects are cyan. The QC gate removes
objects without tissue-high support, but it cannot distinguish every false
positive occurring inside a tissue-high region.


For samples read by `tifffile`, confirm that the crop corresponds exactly to
the tissue shown in notebook 00. The backend changes only image I/O; it must
not change the level-0 coordinate frame.
